In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import optuna
from sklearn.metrics import average_precision_score

# List of query/database images
query_images = []
db_images = []
logo_names = []

# Resize images
img_size = (400, 400)

# Read query images
# Order by name
query_dir = sorted(os.listdir('queries/query/'))
for idx, img_name in enumerate(query_dir):
  # Read image
  img = cv2.imread(os.path.join('queries/query/', img_name))
  # Resize image
  img = cv2.resize(img, dsize=img_size)
  # Save image
  query_images.append(img)
  # Save logo names (remove "_logo.txt" and capitalize)
  logo_names.append(os.path.splitext(img_name)[0][:-5].capitalize())
  #print("Image '" + img_name + "' loaded.")
print("Query images loaded!")

# Read db images
# Order by number
db_dir = sorted(os.listdir('database/'), key=lambda x: int(x.split('.')[0]))
for idx, img_name in enumerate(db_dir):
  # Read image
  img = cv2.imread(os.path.join('database/', img_name))
  # Resize image
  img = cv2.resize(img, dsize=img_size)
  # Save image
  db_images.append(img)
  #print("Image '" + img_name + "' loaded.")
print("Database images loaded!")

# # Ground-truth arrays per logo
coca_cola_GT = np.zeros(len(db_images))
marlboro_GT = np.zeros(len(db_images))
starbucks_GT = np.zeros(len(db_images))
heineken_GT = np.zeros(len(db_images))
coca_cola_GT[18:25] = 1
marlboro_GT[55:61] = 1
marlboro_GT[100:] = 1
starbucks_GT[70:100] = 1
heineken_GT[25:55] = 1
# Matrix containing ground-truth values for each query image
GT_matrix = np.vstack([coca_cola_GT, heineken_GT, marlboro_GT, starbucks_GT])
print(f"GT matrix shape: {GT_matrix.shape}")

n_coca_cola = int(coca_cola_GT.sum())
n_heineken = int(heineken_GT.sum())
n_marlboro = int(marlboro_GT.sum())
n_starbucks = int(starbucks_GT.sum())
n_others = len(db_images) - (n_coca_cola + n_heineken + n_marlboro + n_starbucks)
print(f"Number of database images (coca_cola): {n_coca_cola}")
print(f"Number of database images (heineken): {n_heineken}")
print(f"Number of database images (marlboro): {n_marlboro}")
print(f"Number of database images (starbucks): {n_starbucks}")
print(f"Number of database images (others): {n_others}")


############################## CONFIGURATION PARAMETERS - MODIFY THESE!

# Optuna Optimization Settings
OPTUNA_CONFIG = {
    'n_trials': 1000,                    # Number of optimization trials
    'timeout': None,                    # Maximum time in seconds (None for no timeout)
    'random_state': 42,                 # Random seed for reproducibility
    'n_jobs': 8,                        # Number of parallel jobs (-1 for all cores)
}

# SIFT Parameter Search Space
SIFT_PARAMS = {
    'n_features': [0, 3000],           # [min, max] for n_features
    'n_octaves': [2, 16],              # [min, max] for nOctaveLayers
    'edge_threshold': [2.0, 30.0],     # [min, max] for edgeThreshold
    'contrast_threshold': [0.005, 0.5], # [min, max] for contrastThreshold
    'sigma': [0.5, 5],               # [min, max] for sigma
}

# Matcher Configuration
MATCHER_CONFIG = {
    'types': ['bf', 'flann'],          # Available matcher types
    'lowe_ratio': [0.55, 0.9],         # [min, max] for Lowe's ratio
    'flann_trees': [1, 10],           # [min, max] for FLANN trees
    'flann_checks': [10, 100],        # [min, max] for FLANN checks
}

# Feature Processing Settings
FEATURE_CONFIG = {
    'enable_precise_upscale': True,    # Whether to enable precise upscale
    'cross_check': False,              # Whether to use cross-check in BFMatcher
}

# Evaluation Settings
EVAL_CONFIG = {
    'interpolation_points': 11,        # Number of points for AP interpolation
    'min_positive_samples': 1,         # Minimum positive samples required for AP calculation
}

# Visualization Settings
VISUALIZATION_CONFIG = {
    'plot_matches': True,              # Whether to plot matches
    'match_examples': [40, 60, 80, 109], # Database indices for match examples
    'plot_pr_curves': True,            # Whether to plot precision-recall curves
}

############################## ADVANCED OPTUNA CONFIGURATION

# Advanced optimization settings (modify if you're familiar with Optuna)
ADVANCED_OPTUNA = {
    'sampler': 'TPE',                  # Sampler: 'TPE', 'Random', 'CmaEs'
    'pruner': 'Hyperband',             # Pruner: 'Hyperband', 'Median', 'None'
    'direction': 'maximize',           # Optimization direction
}

# Early Stopping Configuration
EARLY_STOPPING = {
    'enable': True,                    # Enable early stopping
    'patience': 50,                    # Trials without improvement before stopping
    'min_trials': 500,                  # Minimum trials before early stopping can trigger
}

############################## UPDATED OPTUNA OPTIMIZATION FUNCTION

def create_objective_function(config):
    def objective(trial):
        # Suggest SIFT parameters
        n_features = trial.suggest_int('n_features', 
                                     config['SIFT_PARAMS']['n_features'][0],
                                     config['SIFT_PARAMS']['n_features'][1])
        
        n_octaves = trial.suggest_int('n_octaves',
                                    config['SIFT_PARAMS']['n_octaves'][0],
                                    config['SIFT_PARAMS']['n_octaves'][1])
        
        edge_threshold = trial.suggest_float('edge_threshold',
                                           config['SIFT_PARAMS']['edge_threshold'][0],
                                           config['SIFT_PARAMS']['edge_threshold'][1])
        
        contrast_threshold = trial.suggest_float('contrast_threshold',
                                               config['SIFT_PARAMS']['contrast_threshold'][0],
                                               config['SIFT_PARAMS']['contrast_threshold'][1])
        
        sigma = trial.suggest_float('sigma',
                                  config['SIFT_PARAMS']['sigma'][0],
                                  config['SIFT_PARAMS']['sigma'][1])
        
        # Suggest matcher parameters
        matcher_type = trial.suggest_categorical('matcher_type', 
                                               config['MATCHER_CONFIG']['types'])
        
        lowe_ratio = trial.suggest_float('lowe_ratio',
                                       config['MATCHER_CONFIG']['lowe_ratio'][0],
                                       config['MATCHER_CONFIG']['lowe_ratio'][1])
        
        # Create detector
        detector = cv2.SIFT_create(
            nfeatures=n_features,
            nOctaveLayers=n_octaves,
            edgeThreshold=edge_threshold,
            contrastThreshold=contrast_threshold,
            sigma=sigma
        )
        
        # Compute descriptors
        query_des = []
        db_des = []
        
        for img in query_images:
            _, des = detector.detectAndCompute(img, None)
            query_des.append(des)
        
        for img in db_images:
            _, des = detector.detectAndCompute(img, None)
            db_des.append(des)
        
        # Create matcher based on type
        if matcher_type == 'bf':
            matcher = cv2.BFMatcher_create(cv2.NORM_L2, 
                                         crossCheck=config['FEATURE_CONFIG']['cross_check'])
        else:
            flann_trees = trial.suggest_int('flann_trees',
                                          config['MATCHER_CONFIG']['flann_trees'][0],
                                          config['MATCHER_CONFIG']['flann_trees'][1])
            flann_checks = trial.suggest_int('flann_checks',
                                           config['MATCHER_CONFIG']['flann_checks'][0],
                                           config['MATCHER_CONFIG']['flann_checks'][1])
            
            index_params = dict(algorithm=1, trees=flann_trees)  # 1 = FLANN_INDEX_KDTREE
            search_params = dict(checks=flann_checks)
            matcher = cv2.FlannBasedMatcher(index_params, search_params)
        
        # Perform matching
        num_matches = np.zeros((len(query_images), len(db_images)))
        
        for q in range(len(query_images)):
            if query_des[q] is None:
                continue
                
            for db in range(len(db_images)):
                if db_des[db] is None:
                    continue
                
                try:
                    matches = matcher.knnMatch(query_des[q], db_des[db], k=2)
                    n_matches = 0
                    
                    for match_pair in matches:
                        if len(match_pair) == 2:
                            m, n = match_pair
                            if m.distance < lowe_ratio * n.distance:
                                n_matches += 1
                    
                    num_matches[q, db] = n_matches
                except Exception as e:
                    num_matches[q, db] = 0
        
        # Calculate mAP
        AP_all = np.zeros(len(query_images))
        valid_queries = 0
        
        for i in range(len(query_images)):
            scores = num_matches[i]
            gt = GT_matrix[i]
            
            if np.sum(gt) >= config['EVAL_CONFIG']['min_positive_samples']:
                ap = average_precision_score(gt, scores)
                AP_all[i] = ap
                valid_queries += 1
        
        mAP = np.sum(AP_all) / valid_queries if valid_queries > 0 else 0.0
        
        # Store additional information
        trial.set_user_attr('valid_queries', valid_queries)
        trial.set_user_attr('total_features', sum(len(des) if des is not None else 0 for des in query_des + db_des))
        
        return mAP
    
    return objective

############################## RUN OPTIMIZATION WITH CONFIGURATION

print("Starting optimized Optuna study...")

# Combine all configurations
CONFIG = {
    'OPTUNA_CONFIG': OPTUNA_CONFIG,
    'SIFT_PARAMS': SIFT_PARAMS,
    'MATCHER_CONFIG': MATCHER_CONFIG,
    'FEATURE_CONFIG': FEATURE_CONFIG,
    'EVAL_CONFIG': EVAL_CONFIG,
    'ADVANCED_OPTUNA': ADVANCED_OPTUNA,
    'EARLY_STOPPING': EARLY_STOPPING
}

# Create study with advanced configuration
sampler = optuna.samplers.TPESampler(seed=OPTUNA_CONFIG['random_state'])
pruner = optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction=ADVANCED_OPTUNA['direction'],
    sampler=sampler,
    pruner=pruner
)

# Create callback for early stopping
class EarlyStoppingCallback:
    def __init__(self, patience, min_trials):
        self.patience = patience
        self.min_trials = min_trials
        self.best_value = -float('inf')
        self.no_improvement_count = 0
        
    def __call__(self, study, trial):
        if trial.value > self.best_value:
            self.best_value = trial.value
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            
        if (trial.number >= self.min_trials and 
            self.no_improvement_count >= self.patience):
            study.stop()

# Run optimization
early_stopping = EarlyStoppingCallback(
    patience=EARLY_STOPPING['patience'],
    min_trials=EARLY_STOPPING['min_trials']
) if EARLY_STOPPING['enable'] else None

print(f"Running optimization with up to {OPTUNA_CONFIG['n_trials']} trials...")
study.optimize(
    create_objective_function(CONFIG),
    n_trials=OPTUNA_CONFIG['n_trials'],
    timeout=OPTUNA_CONFIG['timeout'],
    n_jobs=OPTUNA_CONFIG['n_jobs'],
    callbacks=[early_stopping] if early_stopping else None
)

print("Optimization completed!")
print(f"Number of completed trials: {len(study.trials)}")

############################## DISPLAY OPTIMIZATION RESULTS

print("\n" + "="*60)
print("OPTIMIZATION RESULTS SUMMARY")
print("="*60)

best_trial = study.best_trial
print(f"Best mAP: {best_trial.value:.4f}")
print(f"Best trial number: {best_trial.number}")

print("\nBest hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

print(f"\nAdditional info:")
print(f"  Valid queries: {best_trial.user_attrs.get('valid_queries', 'N/A')}")
print(f"  Total features: {best_trial.user_attrs.get('total_features', 'N/A')}")

# Plot optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

# Plot parameter importance
try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
except:
    print("Could not plot parameter importance - need more trials")

# Plot parallel coordinate
try:
    fig = optuna.visualization.plot_parallel_coordinate(study)
    fig.show()
except:
    print("Could not plot parallel coordinates - need more trials")

def objective(trial):
    # Suggest hyperparameters for SIFT
    n_features = trial.suggest_int('n_features', 0, 2000)
    n_octaves = trial.suggest_int('n_octaves', 3, 8)
    edge_threshold = trial.suggest_float('edge_threshold', 5.0, 20.0)
    contrast_threshold = trial.suggest_float('contrast_threshold', 0.01, 0.1)
    sigma = trial.suggest_float('sigma', 1.0, 2.0)
    
    # Suggest matcher type and parameters
    matcher_type = trial.suggest_categorical('matcher_type', ['bf', 'flann'])
    lowe_ratio = trial.suggest_float('lowe_ratio', 0.5, 0.85)
    
    # Create detector with suggested parameters
    detector = cv2.SIFT_create(
        nfeatures=n_features,
        nOctaveLayers=n_octaves,
        edgeThreshold=edge_threshold,
        contrastThreshold=contrast_threshold,
        sigma=sigma
    )
    
    # Detect and compute descriptors
    query_des = []
    db_des = []
    
    for i in range(len(query_images)):
        kp, des = detector.detectAndCompute(query_images[i], None)
        query_des.append(des)
    
    for i in range(len(db_images)):
        kp, des = detector.detectAndCompute(db_images[i], None)
        db_des.append(des)
    
    # Choose matcher based on suggestion
    if matcher_type == 'bf':
        # Brute-force matcher with cross-check for better accuracy
        matcher = cv2.BFMatcher_create(cv2.NORM_L2, crossCheck=False)
    else:
        # FLANN matcher - usually more robust
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)
    
    # Matrix for number of matches
    num_matches = np.zeros((len(query_images), len(db_images)))
    
    # For each query image
    for q in range(len(query_images)):
        if query_des[q] is None:
            continue
            
        # For each database image
        for db in range(len(db_images)):
            if db_des[db] is None:
                continue
                
            if matcher_type == 'bf':
                # For BFMatcher, use knnMatch with ratio test
                matches = matcher.knnMatch(query_des[q], db_des[db], k=2)
                # Apply ratio test
                n_matches = 0
                for match_pair in matches:
                    if len(match_pair) == 2:
                        m, n = match_pair
                        if m.distance < lowe_ratio * n.distance:
                            n_matches += 1
                num_matches[q, db] = n_matches
            else:
                # For FLANN, also use ratio test
                matches = matcher.knnMatch(query_des[q], db_des[db], k=2)
                n_matches = 0
                for match_pair in matches:
                    if len(match_pair) == 2:
                        m, n = match_pair
                        if m.distance < lowe_ratio * n.distance:
                            n_matches += 1
                num_matches[q, db] = n_matches
    
    # Calculate mAP
    AP_all = np.zeros(len(query_images))
    valid_queries = 0
    
    for i in range(len(query_images)):
        # Get scores and ground truth for this query
        scores = num_matches[i]
        gt = GT_matrix[i]
        
        # Calculate average precision using sklearn (more robust)
        if np.sum(gt) > 0:  # Only if there are positive examples
            ap = average_precision_score(gt, scores)
            AP_all[i] = ap
            valid_queries += 1
    
    # Return mean AP (handle case where no valid queries)
    if valid_queries > 0:
        mAP = np.sum(AP_all) / valid_queries
    else:
        mAP = 0.0
        
    return mAP

############################## RUN OPTUNA OPTIMIZATION

print("Starting Optuna optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)  # You can increase trials for better results

print("Best trial:")
trial = study.best_trial
print(f"  mAP: {trial.value:.4f}")
print("  Best hyperparameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

############################## FINAL EVALUATION WITH BEST PARAMETERS

print("\n" + "="*50)

Query images loaded!


[I 2025-11-09 17:29:59,754] A new study created in memory with name: no-name-77937fcd-95d1-40b6-8855-f59cf694aff7


Database images loaded!
GT matrix shape: (4, 110)
Number of database images (coca_cola): 7
Number of database images (heineken): 30
Number of database images (marlboro): 16
Number of database images (starbucks): 30
Number of database images (others): 27
Starting optimized Optuna study...
Running optimization with up to 1000 trials...


[I 2025-11-09 17:30:09,203] Trial 2 finished with value: 0.5604658436322515 and parameters: {'n_features': 927, 'n_octaves': 2, 'edge_threshold': 2.056405096875267, 'contrast_threshold': 0.09002645019313292, 'sigma': 1.851042899635196, 'matcher_type': 'bf', 'lowe_ratio': 0.6244463552195201}. Best is trial 2 with value: 0.5604658436322515.
[I 2025-11-09 17:30:11,358] Trial 1 finished with value: 0.20205627705627704 and parameters: {'n_features': 2312, 'n_octaves': 9, 'edge_threshold': 24.561151188255256, 'contrast_threshold': 0.3909647197654469, 'sigma': 1.9190732150666494, 'matcher_type': 'bf', 'lowe_ratio': 0.8859069205731187}. Best is trial 2 with value: 0.5604658436322515.
[I 2025-11-09 17:30:12,801] Trial 6 finished with value: 0.22306186868686867 and parameters: {'n_features': 123, 'n_octaves': 7, 'edge_threshold': 6.095897749955916, 'contrast_threshold': 0.3530391769568577, 'sigma': 2.819471968945826, 'matcher_type': 'bf', 'lowe_ratio': 0.8937951534301845}. Best is trial 2 with v

## Final result

In [ ]:
print("FINAL EVALUATION WITH OPTIMIZED PARAMETERS")
print("="*50)

# Extract best parameters
best_params = trial.params
n_features = best_params['n_features']
n_octaves = best_params['n_octaves']
edge_threshold = best_params['edge_threshold']
contrast_threshold = best_params['contrast_threshold']
sigma = best_params['sigma']
matcher_type = best_params['matcher_type']
lowe_ratio = best_params['lowe_ratio']

# Create detector with optimized parameters
detector = cv2.SIFT_create(
    nfeatures=n_features,
    nOctaveLayers=n_octaves,
    edgeThreshold=edge_threshold,
    contrastThreshold=contrast_threshold,
    sigma=sigma
)

# Detect and compute
query_kps = []
query_des = []
db_kps = []
db_des = []

for i in range(len(query_images)):
    kp, des = detector.detectAndCompute(query_images[i], None)
    query_kps.append(kp)
    query_des.append(des)

for i in range(len(db_images)):
    kp, des = detector.detectAndCompute(db_images[i], None)
    db_kps.append(kp)
    db_des.append(des)

# Create optimized matcher
if matcher_type == 'bf':
    matcher = cv2.BFMatcher_create(cv2.NORM_L2, crossCheck=False)
else:
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    matcher = cv2.FlannBasedMatcher(index_params, search_params)

# Final matching
num_matches = np.zeros((len(query_images), len(db_images)))
logo_matches = []

for q in range(len(query_images)):
    q_matches = []
    for db in range(len(db_images)):
        if query_des[q] is not None and db_des[db] is not None:
            matches = matcher.knnMatch(query_des[q], db_des[db], k=2)
            n_matches = 0
            sel_matches = []
            for match_pair in matches:
                if len(match_pair) == 2:
                    m, n = match_pair
                    if m.distance < lowe_ratio * n.distance:
                        n_matches += 1
                        sel_matches.append(m)
            num_matches[q, db] = n_matches
            q_matches.append(sel_matches)
        else:
            q_matches.append([])
    logo_matches.append(q_matches)

# Calculate final mAP
AP_all = np.zeros(len(query_images))

_ = plt.figure(figsize=(12, 12))
for i in range(len(query_images)):
    scores = num_matches[i]
    gt = GT_matrix[i]
    
    if np.sum(gt) > 0:
        # Calculate precision-recall curve manually for plotting
        idx = np.argsort(-scores)
        TP = gt[idx] == 1
        FP = gt[idx] == 0
        TP = np.cumsum(TP)
        FP = np.cumsum(FP)
        prec = TP / (TP + FP)
        rec = TP / np.sum(gt)
        
        # Interpolated AP
        AP = 0
        interp_prec = []
        for t in np.linspace(0, 1, num=11):
            p = prec[rec >= t]
            if p.size == 0:
                p = 0
            else:
                p = p.max()
            interp_prec.append(p)
            AP = AP + p/11
        AP_all[i] = AP
        
        # Plot
        _ = plt.subplot(2, 2, i+1)
        plt.title(f"{logo_names[i]} - AP: {AP:.3f}")
        _ = plt.plot(rec, prec, '-', linewidth=3)
        _ = plt.plot(np.linspace(0, 1, num=11), interp_prec, 'o', color='red', linewidth=2)
        plt.legend(['Precision', 'Interpolated AP'])
        plt.grid('on')
        plt.xlabel('Recall'), plt.ylabel('Precision')
        plt.axis((0,1,0,1.1))

# Print final results
print('\nOPTIMIZED RESULTS')
print("Average Precision (AP):\n")
for i in range(len(query_images)):
    print(f'{logo_names[i]:9s}  ===> {AP_all[i]:.3f}')

final_mAP = np.mean(AP_all)
print(f"\nMean Average Precision (mAP) ===> {final_mAP:.3f}")